In [ ]:
# ============================================================
# Data Science Term Project - [Evaluation File]
# Prerequisite: Run modeling.ipynb first to generate models.pkl
#
# Evaluation Structure
#   Part 1. K-fold Cross Validation  (required by assignment)
#   Part 2. Hold-out Test Set         (final performance check)
#   Part 3. K-fold vs Hold-out Comparison (model stability check)
# ===========================================

# =====================
# Step 1. Import Libraries
# =====================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')
import pickle

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, make_scorer, silhouette_score
)
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

print('=' * 60)
print('Step 1. Libraries imported successfully')
print('=' * 60)


# ============================================================
# Step 2. Load saved models and data
# ============================================================
# Load serialized model objects, customized thresholds, and preprocessed datasets.
# [Why load from pickle files?]
# To enforce absolute architectural consistency across scripts. By retrieving the exact
# scaler weights, feature orderings, and optimized thresholds calculated in previous stages,
# we guarantee that the evaluation script runs under identical environmental parameters.
MODELS_PATH    = r'../outputs/models.pkl'
PROCESSED_PATH = r'../outputs/processed_data.pkl'

print('\n' + '=' * 60)
print('Step 2. Loading saved models and data')
print('=' * 60)

with open(MODELS_PATH, 'rb') as f:
    saved = pickle.load(f)

scaler         = saved['scaler']
X_test_scaled  = saved['X_test_scaled']
y_test         = saved['y_test']
kmeans         = saved['kmeans']
scaler_km      = saved['scaler_km']
df_cluster     = saved['df_cluster']
risk_labels    = saved['risk_labels']
sil_score      = saved['sil_score']
rf_base        = saved['rf_base']
rf_smote       = saved['rf_smote']
rf_adasyn      = saved['rf_adasyn']
rf_cw          = saved['rf_cw']
rf_km          = saved['rf_km']
best_thresh    = saved['best_thresh']
thr_smote      = saved.get('thr_smote',  0.60)
thr_adasyn     = saved.get('thr_adasyn', 0.62)
thr_cw         = saved.get('thr_cw',     best_thresh)
thr_km         = saved.get('thr_km',     0.86)
feature_names  = saved['feature_names']

print('models.pkl loaded successfully!')
print(f'  Thresholds -> SMOTE={thr_smote:.2f}  ADASYN={thr_adasyn:.2f}  '
      f'class_weight={thr_cw:.2f}  KMeans={thr_km:.2f}')

with open(PROCESSED_PATH, 'rb') as f:
    data = pickle.load(f)

X_train_scaled = data['X_train_scaled']
y_train        = data['y_train']
X              = data['X']

print('processed_data.pkl loaded successfully!')

RF_PARAMS = {
    'n_estimators'    : 300,
    'max_depth'       : 5,
    'min_samples_leaf': 20,
    'random_state'    : 42,
    'n_jobs'          : -1
}


# ============================================================
# Helper functions
# ============================================================
def predict_with_threshold(model, X, threshold=0.5):
    """
    Predict stroke using a custom probability threshold.
    Uses the per-model threshold optimised in modeling.ipynb
    via OOF Balanced Accuracy search instead of default 0.5.
    """
    prob = model.predict_proba(X)[:, 1]
    return (prob >= threshold).astype(int)


holdout_results = {}
kfold_results   = {}


def store_holdout(name, y_true, y_pred):
    """Calculate and store Hold-out evaluation metrics."""
    holdout_results[name] = {
        'Accuracy' : accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall'   : recall_score(y_true, y_pred),
        'F1'       : f1_score(y_true, y_pred),
    }


def print_holdout(name, y_true, y_pred, threshold=None):
    """Print Hold-out results."""
    store_holdout(name, y_true, y_pred)
    r = holdout_results[name]
    thresh_str = f" (Threshold={threshold:.2f})" if threshold else ""
    print(f"\n  Accuracy : {r['Accuracy']:.4f}")
    print(f"  Precision: {r['Precision']:.4f}")
    print(f"  Recall   : {r['Recall']:.4f}  <- Most important in medical diagnosis")
    print(f"  F1-score : {r['F1']:.4f}")


# ============================================================
# Part 1. K-Means Clustering Evaluation
# ============================================================
print('\n\n' + '=' * 60)
print('Part 1. K-Means Clustering Evaluation')
print('=' * 60)
print(f'\nSilhouette Score: {sil_score:.4f}')

# Profile each cluster by calculating mean age, glucose levels, and actual stroke incidence rates.
# [Why perform a comprehensive descriptive profiling on clusters?]
# To validate the clinical relevance of our unsupervised learning phase. By mapping cluster features,
# we discover that the High-Risk group possesses an average glucose level of 205.3 (compared to ~90
# in other groups), empirically proving that glucose is the primary descriptive risk factor for
# stroke categorization.
print('\n[Risk Group Clinical Analysis]')
for cluster_id, label in risk_labels.items():
    group = df_cluster[df_cluster['cluster'] == cluster_id]
    stroke_rate = group['stroke'].mean() * 100
    avg_age     = group['age'].mean()
    avg_glucose = group['avg_glucose_level'].mean()
    print(f'\n  {label} (Cluster {cluster_id}):')
    print(f'    Stroke rate     : {stroke_rate:.1f}%')
    print(f'    Average age     : {avg_age:.1f} years')
    print(f'    Average glucose : {avg_glucose:.1f}')
    print(f'    Sample count    : {len(group)}')


# ============================================================
# Part 2. Evaluation 1 - K-fold Cross Validation (Required)
# ============================================================
# [Why manual fold loop instead of cross_validate]
# cross_validate() always uses predict() internally -> threshold fixed at 0.5.
# We loop over folds manually with predict_proba() + per-model threshold
# so that KFold and Hold-out are evaluated on the same basis.
print('\n\n' + '=' * 60)
print('Part 2. Evaluation 1 - K-fold Cross Validation (Required)')
print('=' * 60)
print('5-fold CV: same per-model threshold as Hold-out evaluation\n')

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def run_kfold(name, pipeline_or_model, X, y, threshold=0.5):
    """
    5-fold CV with per-model probability threshold.
    cross_validate() uses predict() internally (threshold=0.5 fixed).
    Manual fold loop + predict_proba() + threshold ensures KFold
    and Hold-out are evaluated on identical criteria.
    """
    from sklearn.base import clone
    acc_l, prec_l, rec_l, f1_l = [], [], [], []
    for tr_idx, va_idx in SKF.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        m = clone(pipeline_or_model)
        m.fit(X_tr, y_tr)
        prob = m.predict_proba(X_va)[:, 1]
        pred = (prob >= threshold).astype(int)
        acc_l.append(accuracy_score(y_va, pred))
        prec_l.append(precision_score(y_va, pred, zero_division=0))
        rec_l.append(recall_score(y_va, pred))
        f1_l.append(f1_score(y_va, pred))
    kfold_results[name] = {
        'Accuracy' : (np.mean(acc_l),  np.std(acc_l)),
        'Precision': (np.mean(prec_l), np.std(prec_l)),
        'Recall'   : (np.mean(rec_l),  np.std(rec_l)),
        'F1'       : (np.mean(f1_l),   np.std(f1_l)),
    }
    r = kfold_results[name]
    print(f"\n  {'Metric':<12} {'Mean':>8}  {'Std Dev':>8}")
    print(f"  {'-'*32}")
    for metric, (mean, std) in r.items():
        flag = '  <- Most important' if metric == 'Recall' else ''
        print(f"  {metric:<12} {mean:>8.4f}  +/-{std:.4f}{flag}")


# ---------------------------------------------------------------------
# Experiment 1: RF Baseline - K-fold CV
# ---------------------------------------------------------------------
print('-' * 50)
print('Experiment 1: RF Baseline - K-fold CV')
print('-' * 50)
run_kfold('Baseline', RandomForestClassifier(**RF_PARAMS), X_train_scaled, y_train)

# ---------------------------------------------------------------------
# Experiment 2: SMOTE + RF - K-fold CV
# ---------------------------------------------------------------------
# [Why use imblearn.pipeline.Pipeline (ImbPipeline)?]
# Standard pipelines apply preprocessing globally. ImbPipeline ensures that SMOTE
# is executed STRICTLY within the 4 training folds during each cross-validation iteration.
# The 5th validation fold remains completely untouched by synthetic generation,
# preventing data leakage and guaranteeing an honest, unbiased evaluation.
print('\n' + '-' * 50)
print('Experiment 2: SMOTE + RF - K-fold CV')
print('-' * 50)
print('Using ImbPipeline: SMOTE applied only to each fold train split (prevents leakage)')
pipe_smote = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf',    RandomForestClassifier(**RF_PARAMS))
])
run_kfold("SMOTE + RF", pipe_smote, X_train_scaled, y_train, threshold=thr_smote)

# ---------------------------------------------------------------------
# Experiment 3: ADASYN + RF - K-fold CV
# ---------------------------------------------------------------------
print('\n' + '-' * 50)
print('Experiment 3: ADASYN + RF - K-fold CV')
print('-' * 50)
print('Using ImbPipeline: ADASYN applied only to each fold train split')
pipe_adasyn = ImbPipeline([
    ('adasyn', ADASYN(random_state=42)),
    ('rf',     RandomForestClassifier(**RF_PARAMS))
])
run_kfold("ADASYN + RF", pipe_adasyn, X_train_scaled, y_train, threshold=thr_adasyn)

# ---------------------------------------------------------------------
# Experiment 4: class_weight + RF - K-fold CV
# ---------------------------------------------------------------------
print('\n' + '-' * 50)
print(f'Experiment 4: class_weight + RF - K-fold CV  (threshold={thr_cw:.2f})')
print('-' * 50)
run_kfold(
    f"class_weight (Thresh={thr_cw:.2f})",
    RandomForestClassifier(**RF_PARAMS, class_weight='balanced'),
    X_train_scaled, y_train,
    threshold=thr_cw
)

# ---------------------------------------------------------------------
# Experiment 5: K-Means Undersampling + RF - K-fold CV
# ---------------------------------------------------------------------
# Implement a fully customized manual fold iteration specifically for K-Means Undersampling.
# [Why manually loop the K-Means undersampling process across folds?]
# K-Means Undersampling is a custom heuristic that cannot be seamlessly wrapped in standard
# imblearn pipelines. To prevent severe data leakage, we explicitly segment the folds, run
# K-Means clustering strictly on the majority instances of the training split, extract the centroids,
# combine them with the training minority instances, and validate exclusively on the uncorrupted validation split.
print('\n' + '-' * 50)
print(f'Experiment 5: K-Means Undersampling + RF - K-fold CV  (threshold={thr_km:.2f})')
print('-' * 50)
print('Manual fold loop: K-Means undersampling applied to each fold train split only')

kfold_km_scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
X_arr = X_train_scaled.values
y_arr = y_train.values

for fold, (train_idx, val_idx) in enumerate(SKF.split(X_arr, y_arr), 1):
    X_tr, X_val = X_arr[train_idx], X_arr[val_idx]
    y_tr, y_val = y_arr[train_idx], y_arr[val_idx]

    X_maj = X_tr[y_tr == 0]
    X_min = X_tr[y_tr == 1]
    n_min = len(X_min)

    km_fold = KMeans(n_clusters=n_min, random_state=42, n_init=5)
    km_fold.fit(X_maj)
    X_maj_under = km_fold.cluster_centers_
    X_tr_bal = np.vstack([X_maj_under, X_min])
    y_tr_bal = np.array([0] * n_min + [1] * n_min)

    rf_fold = RandomForestClassifier(**RF_PARAMS)
    rf_fold.fit(X_tr_bal, y_tr_bal)

    # Apply thr_km (same threshold as Hold-out)
    prob = rf_fold.predict_proba(X_val)[:, 1]
    pred = (prob >= thr_km).astype(int)

    kfold_km_scores['accuracy'].append(accuracy_score(y_val, pred))
    kfold_km_scores['precision'].append(precision_score(y_val, pred, zero_division=0))
    kfold_km_scores['recall'].append(recall_score(y_val, pred))
    kfold_km_scores['f1'].append(f1_score(y_val, pred))
    print(f'  Fold {fold} done')

kfold_results['K-Means Undersampling + RF'] = {
    'Accuracy' : (np.mean(kfold_km_scores['accuracy']),  np.std(kfold_km_scores['accuracy'])),
    'Precision': (np.mean(kfold_km_scores['precision']), np.std(kfold_km_scores['precision'])),
    'Recall'   : (np.mean(kfold_km_scores['recall']),    np.std(kfold_km_scores['recall'])),
    'F1'       : (np.mean(kfold_km_scores['f1']),        np.std(kfold_km_scores['f1'])),
}
r = kfold_results['K-Means Undersampling + RF']
print(f"\n  {'Metric':<12} {'Mean':>8}  {'Std Dev':>8}")
print(f"  {'-'*32}")
for metric, (mean, std) in r.items():
    flag = '  <- Most important' if metric == 'Recall' else ''
    print(f'  {metric:<12} {mean:>8.4f}  +/-{std:.4f}{flag}')


# ============================================================
# Part 3. Evaluation 2 - Hold-out Test Set (Final Performance)
# ============================================================
# Evaluate model predictions on the 20% independent test split.
# [Why use a Hold-out Test Set as the final evaluation gate?]
# Cross-validation is highly effective for structural tuning, but the hold-out test partition
# represents a simulation of true clinical deployment. Because this data was completely withheld
# during feature fitting, model training, and threshold grid-searches, it serves as the ultimate
# test for generalization.
print('\n\n' + '=' * 60)
print('Part 3. Evaluation 2 - Hold-out Test Set (Final Performance)')
print('=' * 60)
print('Evaluating on 20% test set never seen during training')
print('Using per-model optimised thresholds (same as KFold evaluation)\n')

# Experiment 1: RF Baseline
print('-' * 50)
print('Experiment 1: RF Baseline - Hold-out')
print('-' * 50)
y_pred_base = rf_base.predict(X_test_scaled)
print_holdout('Baseline', y_test, y_pred_base)

# Experiment 2: SMOTE + RF
print('-' * 50)
print(f'Experiment 2: SMOTE + RF - Hold-out  (threshold={thr_smote:.2f})')
print('-' * 50)
y_pred_smote = predict_with_threshold(rf_smote, X_test_scaled, thr_smote)
print_holdout("SMOTE + RF", y_test, y_pred_smote, threshold=thr_smote)

# Experiment 3: ADASYN + RF
print('-' * 50)
print(f'Experiment 3: ADASYN + RF - Hold-out  (threshold={thr_adasyn:.2f})')
print('-' * 50)
y_pred_adasyn = predict_with_threshold(rf_adasyn, X_test_scaled, thr_adasyn)
print_holdout("ADASYN + RF", y_test, y_pred_adasyn, threshold=thr_adasyn)

# Experiment 4: class_weight
print('-' * 50)
print(f'Experiment 4: class_weight + RF - Hold-out  (threshold={thr_cw:.2f})')
print('-' * 50)
y_pred_cw = predict_with_threshold(rf_cw, X_test_scaled, thr_cw)
print_holdout(f'class_weight (Thresh={thr_cw:.2f})', y_test, y_pred_cw, threshold=thr_cw)

# Experiment 5: K-Means Undersampling + RF
print('-' * 50)
print(f'Experiment 5: K-Means Undersampling + RF - Hold-out  (threshold={thr_km:.2f})')
print('-' * 50)
y_pred_km = predict_with_threshold(rf_km, X_test_scaled, thr_km)
print_holdout("K-Means Undersampling + RF", y_test, y_pred_km, threshold=thr_km)


# ============================================================
# Part 4. Evaluation 3 - K-fold vs Hold-out Comparison
# ============================================================
# Calculate the absolute variance between cross-validation metrics and hold-out test metrics.
# [Why perform a stability comparison check?]
# To detect hidden overfitting or data leakage. If a model performs exceptionally well in
# cross-validation but drops significantly (>0.05) on the hold-out set, it signifies a failure to
# generalize. A small delta across both validation paradigms demonstrates robust mathematical stability.
print('\n\n' + '=' * 60)
print('Part 4. Evaluation 3 - K-fold vs Hold-out Comparison (Model Stability)')
print('=' * 60)
print('Both evaluations use the same per-model threshold -> small diff = stable')
print('Large difference -> potential overfitting\n')

exp_names = list(kfold_results.keys())

for name in exp_names:
    kf = kfold_results[name]
    ho = holdout_results.get(name, None)
    print(f'\n  [{name}]')
    print(f"  {'Metric':<12} {'K-fold Mean':>12}  {'Hold-out':>10}  {'Diff':>8}")
    print(f"  {'-'*48}")
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1']:
        kf_mean = kf[metric][0]
        ho_val  = ho[metric] if ho else float('nan')
        diff    = abs(kf_mean - ho_val)
        flag    = '  Warning: large difference' if diff > 0.05 else ''
        print(f'  {metric:<12} {kf_mean:>12.4f}  {ho_val:>10.4f}  {diff:>8.4f}{flag}')


# ============================================================
# Part 5. Save Visualizations
# ============================================================
# Generate, format, and export 5 distinct evaluation plots to the disk.
# [Why automate plot generation and saving?]
# To transform abstract mathematical arrays into highly intuitive visual evidence.
# These charts provide a clear narrative for presentations, directly proving
# how our custom K-Means Undersampling architecture maximizes diagnostic Recall while
# maintaining structural stability compared to baseline models.
print('\n\n' + '=' * 60)
print('Part 5. Saving Visualizations...')
print('=' * 60)

SAVE_DIR = r'../outputs'

# Plot 1: K-fold Recall comparison
fig, ax = plt.subplots(figsize=(12, 5))
names   = list(kfold_results.keys())
recalls = [kfold_results[n]['Recall'][0] for n in names]
stds    = [kfold_results[n]['Recall'][1] for n in names]
colors  = ['tomato' if v == max(recalls) else 'steelblue' for v in recalls]
bars = ax.bar(names, recalls, color=colors, alpha=0.85, yerr=stds,
              capsize=5, error_kw={'elinewidth': 2})
ax.set_ylabel('Recall')
ax.set_title('K-fold CV Recall Comparison (Mean +/- Std Dev)\n(Same per-model threshold applied)')
ax.set_ylim(0, 1.15)
for bar, val, std in zip(bars, recalls, stds):
    ax.text(bar.get_x() + bar.get_width()/2, val + std + 0.03,
            f'{val:.3f}', ha='center', fontweight='bold', fontsize=9)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/kfold_recall.png', dpi=150)
plt.close()
print('Saved: kfold_recall.png')

# Plot 2: All metrics comparison
metrics  = ['Accuracy', 'Precision', 'Recall', 'F1']
x        = np.arange(len(names))
width    = 0.18
colors_m = ['steelblue', 'orange', 'tomato', 'mediumseagreen']
fig, ax = plt.subplots(figsize=(15, 6))
for i, metric in enumerate(metrics):
    vals = [kfold_results[n][metric][0] for n in names]
    errs = [kfold_results[n][metric][1] for n in names]
    ax.bar(x + i * width, vals, width, label=metric,
           color=colors_m[i], alpha=0.85, yerr=errs, capsize=3)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('K-fold CV All Metrics Comparison (Mean +/- Std Dev)')
ax.legend(loc='upper right')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/kfold_all_metrics.png', dpi=150)
plt.close()
print('Saved: kfold_all_metrics.png')

# Plot 3: K-fold vs Hold-out Recall comparison
fig, ax = plt.subplots(figsize=(12, 5))
x       = np.arange(len(exp_names))
width   = 0.35
kf_recalls = [kfold_results[n]['Recall'][0] for n in exp_names]
kf_stds    = [kfold_results[n]['Recall'][1] for n in exp_names]
ho_recalls = [holdout_results[n]['Recall']   for n in exp_names]
ax.bar(x - width/2, kf_recalls, width, label='K-fold CV',
       color='steelblue', alpha=0.85, yerr=kf_stds, capsize=4)
ax.bar(x + width/2, ho_recalls, width, label='Hold-out',
       color='tomato', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(exp_names, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Recall')
ax.set_title('K-fold vs Hold-out Recall Comparison\n(Same threshold applied -> smaller gap = more stable)')
ax.legend()
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/kfold_vs_holdout.png', dpi=150)
plt.close()
print('Saved: kfold_vs_holdout.png')

# Plot 4: Confusion Matrix comparison (5 models)
all_preds  = [y_pred_base, y_pred_smote, y_pred_adasyn, y_pred_cw, y_pred_km]
titles     = ['Baseline', 'SMOTE+RF', 'ADASYN+RF',
              f'class_weight\nThresh={thr_cw:.2f}', 'K-Means\nUndersampling+RF']
fig, axes  = plt.subplots(1, 5, figsize=(22, 5))
for ax, pred, title in zip(axes, all_preds, titles):
    cm = confusion_matrix(y_test, pred)
    im = ax.imshow(cm, interpolation='nearest', cmap='viridis')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Normal', 'Stroke'])
    ax.set_yticklabels(['Normal', 'Stroke'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white', fontsize=14, fontweight='bold')
fig.suptitle('Confusion Matrix Comparison (Hold-out Test Set)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: confusion_matrices.png')

# Plot 5: Feature importance
fig, ax = plt.subplots(figsize=(10, 6))
imp = pd.Series(rf_cw.feature_importances_, index=feature_names).sort_values()
top = imp.tail(10)
colors_imp = ['tomato' if n == 'age' else 'steelblue' for n in top.index]
ax.barh(top.index, top.values, color=colors_imp)
ax.set_xlabel('Feature Importance')
ax.set_title('Feature Importance Top 10 (class_weight RF)\nAge dominates predictive power')
for i, v in enumerate(top.values):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/feature_importance.png', dpi=150)
plt.close()
print('Saved: feature_importance.png')

print('\nAll visualizations saved!')
print('\nEvaluation complete.')


Step 1. Libraries imported successfully

Step 2. Loading saved models and data
models.pkl loaded successfully!
  Thresholds -> SMOTE=0.62  ADASYN=0.62  class_weight=0.56  KMeans=0.76
processed_data.pkl loaded successfully!


Part 1. K-Means Clustering Evaluation

Silhouette Score: 0.3719

[Risk Group Clinical Analysis]

  Low Risk (Cluster 0):
    Stroke rate     : 0.2%
    Average age     : 18.8 years
    Average glucose : 92.7
    Sample count    : 1712

  Medium Risk (Cluster 2):
    Stroke rate     : 5.7%
    Average age     : 54.2 years
    Average glucose : 89.1
    Sample count    : 2699

  High Risk (Cluster 1):
    Stroke rate     : 13.0%
    Average age     : 60.4 years
    Average glucose : 205.3
    Sample count    : 698


Part 2. Evaluation 1 - K-fold Cross Validation (Required)
5-fold CV: same per-model threshold as Hold-out evaluation

--------------------------------------------------
Experiment 1: RF Baseline - K-fold CV
------------------------------------------------